# Capítulo 1: Explorando a classe `GDALDataset`

**Chris Holden (ceholden@gmail.com) - [https://github.com/ceholden](https://github.com/ceholden)**

-----

## Configuração do Colab (Repetição do Capítulo 0)

Para garantir que o ambiente esteja pronto, vamos instalar as bibliotecas novamente. Se esta célula já foi executada na mesma sessão, ela será rápida.

In [ ]:
# Instala o GDAL e o Matplotlib.
!pip install gdal matplotlib --quiet

## Introdução à `GDALDataset`

Um dos componentes mais fundamentais da biblioteca raster GDAL é uma "[classe](https://en.wikipedia.org/wiki/Class_\(computer_programming\))", ou um objeto, que armazena todas as informações que se possa desejar sobre uma imagem raster. Esta classe, a `GDALDataset`, combina as informações sobre um arquivo raster com as ações que se pode querer executar com ele, como a leitura da imagem.

As informações armazenadas por uma classe são geralmente chamadas de "[propriedades](https://en.wikipedia.org/wiki/Property_\(programming\))" ou atributos, e as ações que uma classe pode executar são chamadas de "[métodos](https://en.wikipedia.org/wiki/Method_\(computer_programming\))".

Para aqueles que buscam uma referência completa e detalhada da "Application Programming Interface" ([API](http://en.wikipedia.org/wiki/Application_programming_interface)) do GDAL, a documentação está disponível online.

Alguns métodos da classe `GDALDataset` incluem:

  * `GetDriver`
  * `GetRasterBand`
  * `GetGeoTransform`
  * `GetProjection`
  * `GetSubDatasets`

Estes métodos são chamados de métodos "getter" (obtener) que permitem acessar os atributos da classe. Por exemplo, ao chamar `GetDriver`, o *dataset* GDAL retorna o *driver* de formato de imagem (ex: ENVI, GeoTIFF, HDF) responsável pelas operações de entrada e saída. De forma similar, `GetGeoTransform` retorna a transformação usada para converter coordenadas de pixel em coordenadas de projeção.

> **Nota sobre o design da API:** Os métodos "getter" e "setter" para acessar e definir propriedades da classe **não são** considerados uma prática "Pythonica". Eles existem porque a API foi originalmente escrita em C++, onde tais métodos são padrão.

Outros métodos importantes permitem definir atributos, os "setter" métodos, incluindo:

  * `SetGeoTransform`
  * `SetProjection`

Esses métodos permitem modificar a projeção geográfica e a localização da imagem.

## Importação de Módulos em Python

Para começar, precisamos importar o GDAL e o NumPy (para manipulação de dados em *array*). Faremos isso usando instruções `import` para trazer as funções, classes e variáveis para o nosso *namespace* (espaço de nomes).

In [ ]:
# Importa o submódulo "gdal" de "osgeo" (prática padrão no Python 3)
from osgeo import gdal

# Importa o numpy, a base do cálculo científico (usando o alias 'np')
import numpy as np

# A importação do 'from __future__ import print_function' (Python 2)
# foi removida, pois estamos usando o Python 3 nativamente.

# Podemos verificar a versão que estamos executando
print("Versão do GDAL: " + gdal.__version__)
print(gdal)

Uma vez que importamos o submódulo `gdal` e `numpy`, precisamos referenciar o caminho completo (ex: `gdal.GDT_Byte` ou `np.array`) ao acessar classes, variáveis ou funções, já que elas não existem no *namespace* global.

In [ ]:
# Vamos imprimir o valor do tipo de dado Byte do GDAL (GDT_Byte)
# O número impresso é como o GDAL rastreia os vários tipos de dados
# Este é o que é chamado de tipo enumerado, ou 'enum'.

# Funciona: acessando através do módulo gdal
print(gdal.GDT_Byte)

# Não funciona (exemplo do erro, já que não está no namespace global)
try:
    print(GDT_Byte)
except NameError as e:
    print(f"\nErro esperado: {e}")

Com esta explicação sobre o *namespace* do Python, vamos aos exemplos.

### Exemplos

#### Abrir uma imagem

Ao abrir uma imagem no GDAL, criamos um objeto `GDALDataset`. Abrimos a imagem com a função `Open` dentro do módulo `gdal`.

Usaremos uma imagem de exemplo (simulada aqui por um caminho local) para este capítulo. Esta imagem é um subconjunto de uma imagem Landsat 7 contendo 7 bandas.

> **Atenção (Adaptação para Colab):** Como o Google Colab não tem acesso ao sistema de arquivos local do autor (`../../example/...`), você deve instruir seus alunos a **carregar um arquivo raster** para a sessão do Colab ou usar um **caminho de arquivo de exemplo** que eles configuraram (ex: `'/content/meu_arquivo.tif'`).

In [ ]:
# Substitua o caminho abaixo pelo caminho do seu arquivo raster no Colab.
# Exemplo: dataset = gdal.Open('/content/LE70220491999322EDC01_stack.gtif', gdal.GA_ReadOnly)

# Para simular o código rodando em um ambiente de desenvolvimento sem o arquivo,
# vamos apenas criar um objeto None para evitar erros neste notebook.
# SE VOCÊ TIVER O ARQUIVO, USE A LINHA ABAIXO (e comente a linha 'dataset = None'):
# dataset = gdal.Open('caminho_para_sua_imagem.tif', gdal.GA_ReadOnly)
dataset = gdal.Open('https://github.com/Alixandrini/ENGJ20/raw/refs/heads/main/Aula%204/LE70220491999322EDC01_stack.gtif', gdal.GA_ReadOnly)

# Verificação para evitar erros no Colab se o arquivo não for fornecido:
if dataset is None:
    print("Atenção: Nenhum dataset foi aberto. Certifique-se de que o caminho do arquivo está correto no Colab.")
else:
    print(f"Dataset aberto: {dataset}")

# Nota: O resto dos exemplos assumirá que 'dataset' foi aberto corretamente.

#### Atributos da Imagem

Agora que temos o *dataset* (assumindo que o arquivo foi carregado e aberto), vamos explorar algumas de suas capacidades.

In [ ]:
if dataset is not None:
    # Quantas bandas esta imagem tem?
    num_bands = dataset.RasterCount
    print('Número de bandas na imagem: {n}\n'.format(n=num_bands))

    # Quantas linhas e colunas?
    rows = dataset.RasterYSize
    cols = dataset.RasterXSize
    print('Tamanho da imagem: {r} linhas x {c} colunas\n'.format(r=rows, c=cols))

    # A imagem possui descrição ou metadados?
    desc = dataset.GetDescription()
    metadata = dataset.GetMetadata()
    print('Descrição Raster: {desc}'.format(desc=desc))
    print('Metadados Raster:')
    print(metadata)
    print('\n')

    # Qual driver foi usado para abrir o raster?
    driver = dataset.GetDriver()
    print('Driver Raster: {d}\n'.format(d=driver.ShortName))

    # Qual é a projeção do raster?
    proj = dataset.GetProjection()
    print('Projeção da Imagem:')
    print(proj + '\n')

    # Qual é a "geo-transformação" do raster?
    gt = dataset.GetGeoTransform()
    print('Geo-transformação da Imagem: {gt}\n'.format(gt=gt))

As primeiras informações obtidas são diretas: tamanho do raster, número de bandas, descrição, metadados e formato de arquivo.

A projeção da imagem está formatada no que é conhecido como "Well Known Text" (WKT). O último dado acessado, a "geotransformação" (`gt`), é um conjunto de 6 números que fornecem todas as informações necessárias para transformar entre coordenadas de pixel e coordenadas projetadas.

Em ordem, os 6 números são:

  * **gt[0]** - Coordenada X da borda superior esquerda do pixel superior esquerdo (coordenada de origem X)
  * **gt[1]** - Resolução no Eixo X (largura do pixel)
  * **gt[2]** - Coeficiente de rotação (0 para imagens North-Up)
  * **gt[3]** - Coordenada Y da borda superior esquerda do pixel superior esquerdo (coordenada de origem Y)
  * **gt[4]** - Coeficiente de rotação (0 para imagens North-Up)
  * **gt[5]** - Resolução no Eixo Y (altura do pixel, geralmente negativa para imagens "North Up")

#### Bandas Raster da Imagem

O objeto `GDALDataset` não é usado diretamente para ler a imagem raster na memória; precisamos acessar cada banda individualmente usando o método `GetRasterBand`. O índice das bandas no GDAL **começa em 1**.

In [ ]:
if dataset is not None:
    # Abre a banda 1 (azul, por exemplo, na ordem padrão Landsat)
    blue = dataset.GetRasterBand(1)
    print(blue)

Vamos explorar alguns dos atributos e métodos da `GDALRasterBand`:

In [ ]:
if dataset is not None:
    # Qual é o tipo de dado da banda?
    datatype = blue.DataType
    print('Tipo de dado da Banda (enum): {dt}'.format(dt=blue.DataType))

    # Podemos obter o nome útil para este tipo enumerado (enum)
    datatype_name = gdal.GetDataTypeName(blue.DataType)
    print('Nome do Tipo de dado da Banda: {dt}'.format(dt=datatype_name))

    # Também podemos perguntar quanto espaço este tipo de dado ocupa
    bytes_size = gdal.GetDataTypeSize(blue.DataType) // 8  # Divide por 8 para obter bytes
    print('Tamanho do tipo de dado da Banda: {b} bytes\n'.format(b=bytes_size))

    # E algumas estatísticas da banda?
    # Os argumentos (0, 1) significam 'recalcular' (0) ou 'usar cache' (1)
    band_min, band_max, band_mean, band_stddev = blue.GetStatistics(0, 1)
    print('Intervalo da Banda: {minimum} - {maximum}'.format(maximum=band_max,
                                                           minimum=band_min))
    print('Média, Desvio Padrão da Banda: {m}, {s}\n'.format(m=band_mean, s=band_stddev))

É importante notar que não precisamos ler a imagem na memória do Python para calcular essas estatísticas - o GDAL fez isso por nós.

Para a maioria das aplicações, precisaremos ler nossas bandas raster na memória. Faremos isso em um *array* 2D do **NumPy**. O NumPy é o "pacote fundamental para computação científica com Python", permitindo-nos representar dados de maneira muito eficiente em termos de memória.

Já importamos o NumPy como `np`, prática comum:

In [ ]:
# Reafirma a importação do NumPy e verifica a versão
import numpy as np
print(f"Versão do NumPy: {np.__version__}")

Para ler nossa banda em um desses objetos `np.array`, usaremos o método `ReadAsArray` do nosso objeto `GDALRasterBand`.

In [ ]:
if dataset is not None:
    # Exibe a documentação do método ReadAsArray
    help(blue.ReadAsArray)

O método `ReadAsArray` aceita argumentos que nos permitem especificar um subconjunto da imagem raster (usando *offsets* X e Y e tamanhos). Lembre-se desta capacidade ao processar imagens grandes ou trabalhar com memória limitada.

Por enquanto, leremos a imagem inteira:

In [ ]:
if dataset is not None:
    blue_data = blue.ReadAsArray()
    print(blue_data)
    print()
    print('Média da banda Azul (calculada via NumPy): {m}'.format(m=blue_data.mean()))
    print('Tamanho (shape) do array: {sz}'.format(sz=blue_data.shape))

Com os dados lidos em um *array* NumPy, podemos realizar operações estatísticas e de álgebra linear.

Se quisermos ler todas as bandas em um único *dataset* 3D (linhas x colunas x bandas), faremos o seguinte:

In [ ]:
if dataset is not None:
    # Inicializa um array 3D -- usando as propriedades de tamanho para portabilidade
    # O dtype será o padrão (float64, geralmente) se não for especificado
    image = np.zeros((dataset.RasterYSize, dataset.RasterXSize, dataset.RasterCount))

    # Percorre todas as bandas no dataset
    for b in range(dataset.RasterCount):
        # Lembre-se: o índice do GDAL é 1-based, o Python é 0-based
        # Portanto, adicionamos 1 na chamada do GDAL
        band = dataset.GetRasterBand(b + 1)

        # Lê os dados da banda na terceira dimensão (índice 'b') do nosso array
        image[:, :, b] = band.ReadAsArray()

    print(f"Array de Imagem 3D:\n{image}")
    print(f"Tipo de dado (dtype) do array: {image.dtype}")

### Otimização de Tipo de Dados

Uma melhoria importante é garantir que o *array* NumPy criado tenha o mesmo tipo de dado que a imagem GDAL original, o que economiza memória. O GDAL fornece uma função para essa tradução `GDAL` \<-\> `NumPy`.

In [ ]:
if dataset is not None:
    from osgeo import gdal_array

    # Obtém o tipo de dado da primeira banda
    image_datatype = dataset.GetRasterBand(1).DataType

    # Aloca nosso array, mas de forma mais eficiente
    image_correct = np.zeros((dataset.RasterYSize, dataset.RasterXSize, dataset.RasterCount),
                     dtype=gdal_array.GDALTypeCodeToNumericTypeCode(image_datatype))

    # Percorre todas as bandas
    for b in range(dataset.RasterCount):
        band = dataset.GetRasterBand(b + 1)
        image_correct[:, :, b] = band.ReadAsArray()

    print("Comparação de Tipos de Dados: ")
    print("    Quando não especificado: {dt}".format(dt=image.dtype)) # 'image' deve ser float64 ou similar
    print("    Quando especificado: {dt}".format(dt=image_correct.dtype)) # Ex: uint16
    print("Isso economiza memória!")

### Fechando o Dataset

O último conceito importante é como desalocar a memória e fechar os *datasets*. Para fechar seus *datasets* GDAL e sinalizar que seus *arrays* NumPy podem ser desalocados (liberados da memória), você pode simplesmente atribuir-lhes o valor `None`:

In [ ]:
# Fecha o dataset GDAL
dataset = None

# Libera os arrays NumPy da memória
image = None
image_correct = None

print("Datasets e Arrays liberados.")